# 06 · Tự động hóa vận hành: nạp dữ liệu hằng ngày, báo cáo Excel, dataset cho analyst

| Yêu cầu công việc | Phần trong project |
|---|---|
| *Automate repetitive reporting using Python, SQL, Excel or AI tools* | Pipeline tuần tự sinh **file Excel 7 sheet** (tóm tắt + 5 phòng ban + chất lượng dữ liệu) — `src/excel_report.py` |
| *Support data extraction and operational reporting for production teams* | **Nạp file dữ liệu mỗi ngày** + **12 kiểm tra chất lượng**, lỗi thì dừng & cách ly; log vận hành theo ngày — `src/ingest.py`, `src/quality.py` |
| *Prepare datasets for analysts and product teams* | **4 bảng dữ liệu sạch** + **từ điển dữ liệu tự sinh** — `src/marts.py` |

Notebook chạy trên thư mục demo riêng (`data/_demo/`) để không ảnh hưởng kho chính.

In [1]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))  # để import được thư mục src/

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import seaborn as sns

pd.set_option("display.max_columns", None)
plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False})
BLUE, RED, GREY = "#2f6db5", "#d1495b", "#9aa5b1"

In [2]:
import os
os.environ.setdefault("LOKY_MAX_CPU_COUNT", "4")
os.environ.setdefault("LLM_MODE", "off")
from src import ingest, marts, pipeline, quality
from src.config import DATA_DIR

demo = ingest.Paths(root=DATA_DIR / "_demo", reports=DATA_DIR / "_demo" / "reports")
ingest.reset(demo)

## 1. Nạp dữ liệu hằng ngày có kiểm tra chất lượng
### 1a. Mô phỏng hệ thống đổ file mỗi ngày
Chia `ticket_history.csv` thành 1 file / ngày (giữ nguyên dữ liệu gốc, kể cả dòng trùng) — giả lập "hôm nay là 07/03/2022" nên chỉ để lại file tới 06/03/2022.

In [3]:
n_files = ingest.simulate_drops(demo)
for f in demo.incoming.glob("*.csv"):
    if ingest.file_date(f) > "2022-03-06":
        f.unlink()
print(f"Đã tạo {n_files} file; còn {len(list(demo.incoming.glob('*.csv')))} file tới 06/03/2022")

Đã tạo 1174 file; còn 874 file tới 06/03/2022


### 1b. 12 kiểm tra trước khi nạp
`error` = dừng, không nạp, không chạy báo cáo · `warning` = vẫn nạp, ghi lại để theo dõi.

In [4]:
checks = pd.DataFrame([
    ("schema", "error", "Đủ 12 cột bắt buộc"),
    ("not_empty", "error", "File có dữ liệu"),
    ("parse_types", "error", "time / giá / id đọc được đúng kiểu"),
    ("key_nulls", "error", "Không thiếu ticket_id, customer_id, time, status_id, paying_method"),
    ("exact_duplicates", "warning", "Dòng trùng hoàn toàn → tự loại bỏ"),
    ("ticket_id_conflict", "error", "Cùng ticket_id nhưng nội dung khác nhau"),
    ("already_loaded", "error", "ticket_id đã có trong kho (chống nạp 2 lần)"),
    ("unknown_status / customer / campaign", "error", "Mã không có trong bảng tham chiếu"),
    ("unknown_device / paying_method", "warning", "Giá trị lạ nhưng không làm sai chỉ số chính"),
    ("price_rules", "error", "Giá không âm, giảm giá ≤ giá gốc, final = gốc − giảm"),
    ("date_matches_file", "error", "Mọi dòng đúng ngày của file"),
    ("volume", "warning", f"Số dòng trong khoảng {quality.VOLUME_LOW}–{quality.VOLUME_HIGH} lần trung vị 28 ngày"),
], columns=["check", "mức độ", "ý nghĩa"])
checks

,check,mức độ,ý nghĩa
0,schema,error,Đủ 12 cột bắt buộc
1,not_empty,error,File có dữ liệu
2,parse_types,error,time / giá / id đọc được đúng kiểu
3,key_nulls,error,"Không thiếu ticket_id, customer_id, time, stat..."
4,exact_duplicates,warning,Dòng trùng hoàn toàn → tự loại bỏ
5,ticket_id_conflict,error,Cùng ticket_id nhưng nội dung khác nhau
6,already_loaded,error,ticket_id đã có trong kho (chống nạp 2 lần)
7,unknown_status / customer / campaign,error,Mã không có trong bảng tham chiếu
8,unknown_device / paying_method,warning,Giá trị lạ nhưng không làm sai chỉ số chính
9,price_rules,error,"Giá không âm, giảm giá ≤ giá gốc, final = gốc ..."


File cập nhật bảng tham chiếu (khách hàng, campaign, thiết bị, mã lỗi — mục 1e) có bộ kiểm tra riêng: đủ cột, khóa không trống, đọc được kiểu dữ liệu, không trùng khóa mâu thuẫn, giá trị phân loại hợp lệ, và ghi lại bản ghi cũ bị đổi thông tin.

Các ngưỡng được đặt **sau khi kiểm tra dữ liệu thật**: dữ liệu 2019–2022 đạt 0 vi phạm ở mọi check `error`. Riêng `volume`: số dòng/ngày dao động 1 → 1,268 theo lịch chiếu phim; ngưỡng 0.2–5 lần cảnh báo tới ~11% số ngày (quá nhiễu), nên chọn 0.1–10 lần (~5%, chủ yếu các đợt rạp mở lại sau COVID).

### 1c. Chạy nạp — chỉ đọc file MỚI

In [5]:
result = ingest.run(demo)
print({k: v for k, v in result.items() if k != "warnings"})
print("Chạy lại lần 2 (không có file mới):", ingest.run(demo)["loaded"], "file")

{'loaded': 874, 'rows': 65729, 'reference_updates': {}, 'failed': None, 'failed_checks': [], 'split_exports': [], 'unrecognized_files': [], 'last_loaded_date': '2022-03-06'}


Chạy lại lần 2 (không có file mới): 0 file


In [6]:
log = pd.read_csv(demo.dq_dir / "dq_log.csv")
warning_counts = log["warnings"].dropna().str.split(";").explode().value_counts()
print(f"{len(log)} file đã kiểm tra · {(log['status'] == 'failed').sum()} file bị chặn · {log['warnings'].notna().sum()} file có cảnh báo")
warning_counts.rename("số file có cảnh báo").to_frame()

874 file đã kiểm tra · 0 file bị chặn · 63 file có cảnh báo


,số file có cảnh báo
warnings,
volume,55
unknown_device,11
exact_duplicates,1


Log vận hành theo ngày (đội vận hành mở mỗi sáng):

In [7]:
pd.read_csv(demo.daily_ops).tail(7)

,date,tickets,success,success_rate,revenue,errors_customer,errors_external,errors_internal
867,2022-02-28,51,48,0.9412,327.81,1,2,0
868,2022-03-01,39,33,0.8462,281.19,2,4,0
869,2022-03-02,67,56,0.8358,416.38,0,11,0
870,2022-03-03,147,115,0.7823,919.65,2,30,0
871,2022-03-04,220,174,0.7909,1368.81,1,45,0
872,2022-03-05,282,229,0.8121,1724.64,3,47,3
873,2022-03-06,213,168,0.7887,1208.30,3,42,0


### 1d. File lỗi → pipeline DỪNG, không ra báo cáo sai
Tạo file ngày 07/03/2022 có 4 loại lỗi thường gặp: mã trạng thái lạ, thiếu customer_id, giá sai công thức, trùng ticket_id.

In [8]:
bad = ingest.inject_bad_file(demo)
try:
    pipeline.run(ingest_first=True, paths=demo)
except pipeline.DataQualityError as exc:
    print("⛔", exc)

⛔ File ticket_history_2022-03-07.csv không đạt kiểm tra chất lượng ['key_nulls', 'ticket_id_conflict', 'unknown_status', 'unknown_customer', 'price_rules'] - đã cách ly vào D:\job\project movie ticket\data\_demo\quarantine, KHÔNG chạy báo cáo.


In [9]:
import json
report = json.loads((demo.dq_dir / f"{bad.stem}.json").read_text(encoding="utf-8"))
pd.DataFrame(report["checks"]).query("status == 'fail'")[["check", "severity", "failed_rows", "detail"]]

,check,severity,failed_rows,detail
3,key_nulls,error,2,customer_id: 2
5,ticket_id_conflict,error,2,cùng ticket_id nhưng nội dung khác nhau
7,unknown_status,error,3,mã lạ: [99]
8,unknown_customer,error,2,customer_id không có trong bảng customer
12,price_rules,error,1,giá âm / giảm giá > giá gốc / final ≠ gốc - giảm


In [10]:
print("Thư mục cách ly:", [f.name for f in demo.quarantine.iterdir()])
print("Trạng thái kho:", json.dumps({k: v for k, v in ingest.read_state(demo).items() if k == "failed"}, ensure_ascii=False))

Thư mục cách ly: ['ticket_history_2022-03-07.csv']
Trạng thái kho: {"failed": {"ticket_history_2022-03-07.csv": {"kind": "ticket_history", "date": "2022-03-07", "errors": ["key_nulls", "ticket_id_conflict", "unknown_status", "unknown_customer", "price_rules"]}}}


File lỗi bị chuyển sang `quarantine/`, kho vẫn giữ nguyên dữ liệu sạch tới 06/03. Khi nguồn gửi lại file đã sửa vào `incoming/`, lần chạy sau sẽ tự nạp tiếp. (File nạp thành công được chuyển sang `archive/`, nên `incoming/` chỉ còn file đang chờ.)

### 1e. Dữ liệu mới ngoài đời: khách mới, campaign mới, mã lỗi mới, file xuất nhiều ngày
Ngày nào cũng có khách mới đăng ký, campaign mới… nên ngoài file giao dịch, nguồn gửi kèm **file cập nhật bảng tham chiếu** cùng quy ước tên: `customer_YYYY-MM-DD.csv`, `campaign_…`, `device_detail_…`, `status_detail_…`. Trong cùng 1 ngày, bảng tham chiếu được kiểm tra & cập nhật **trước**, rồi mới kiểm tra giao dịch → khách đăng ký và mua vé cùng ngày vẫn qua.

Bỏ file lỗi ở trên (coi như nguồn đã gửi bản đúng), rồi thử lần lượt:

In [11]:
(demo.quarantine / bad.name).unlink()
state = ingest.read_state(demo)
state["failed"].clear()
ingest.write_state(demo, state)

template = pd.read_parquet(demo.warehouse / "2022-03-05.parquet")


def new_day(date, n=200, new_customers=(), campaign=None, status=None, tag=""):
    d = template.head(n).copy()
    d["ticket_id"] = [f"n{date.replace('-', '')}{tag}{i:05d}" for i in range(len(d))]
    d["time"] = date + pd.to_datetime(d["time"]).dt.strftime(" %H:%M:%S.%f").str[:-3]
    for i, cid in enumerate(new_customers):
        d.loc[d.index[i], "customer_id"] = cid
    if campaign:
        d.loc[d.index[:3], "campaign_id"] = campaign
    if status:
        d.loc[d.index[:2], "status_id"] = status
    return d


steps = []
def step(label):
    r = ingest.run(demo)
    days = r["split_exports"][0]["days"] if r["split_exports"] else ""
    steps.append((label, r["loaded"], r["reference_updates"] or "", days, r["failed"] or "", ", ".join(r["failed_checks"])))

new_ids = [900001, 900002, 900003]
new_day("2022-03-07", new_customers=new_ids).to_csv(demo.incoming / "ticket_history_2022-03-07.csv", index=False)
step("07/03: có 3 khách mới, nguồn QUÊN gửi file customer")

pd.DataFrame({"customer_id": new_ids, "usergender": ["Female", "Male", "Female"],
              "dob": ["2001-04-12", "1998-11-30", "2003-01-05"]}).to_csv(demo.incoming / "customer_2022-03-07.csv", index=False)
(demo.quarantine / "ticket_history_2022-03-07.csv").rename(demo.incoming / "ticket_history_2022-03-07.csv")
step("07/03: gửi bổ sung customer_2022-03-07.csv")

new_day("2022-03-08", campaign=555555, status=-8).to_csv(demo.incoming / "ticket_history_2022-03-08.csv", index=False)
pd.DataFrame({"campaign_id": [555555], "campaign_type": ["voucher"]}).to_csv(demo.incoming / "campaign_2022-03-08.csv", index=False)
pd.DataFrame({"status_id": [-8], "description": ["3-D Secure timeout"], "error_group": ["bank"]}).to_csv(
    demo.incoming / "status_detail_2022-03-08.csv", index=False)
step("08/03: campaign mới + mã lỗi mới -8 khai nhóm lỗi 'bank' (sai)")

pd.DataFrame({"status_id": [-8], "description": ["3-D Secure timeout"], "error_group": ["external"]}).to_csv(
    demo.incoming / "status_detail_2022-03-08__fixed.csv", index=False)
step("08/03: gửi lại status_detail đã sửa nhóm lỗi = external")

pd.concat([new_day(d, n=60, tag="x") for d in ["2022-03-09", "2022-03-10"]]).to_csv(
    demo.incoming / "ticket_history_export_09-10.csv", index=False)
step("09-10/03: nhận 1 file xuất 2 ngày")

pd.DataFrame(steps, columns=["tình huống", "file giao dịch nạp", "cập nhật tham chiếu", "tách thành (ngày)",
                             "file bị chặn", "lý do"])

,tình huống,file giao dịch nạp,cập nhật tham chiếu,tách thành (ngày),file bị chặn,lý do
0,"07/03: có 3 khách mới, nguồn QUÊN gửi file cus...",0,,,ticket_history_2022-03-07.csv,unknown_customer
1,07/03: gửi bổ sung customer_2022-03-07.csv,1,{'customer': 3},,,
2,08/03: campaign mới + mã lỗi mới -8 khai nhóm ...,0,{'campaign': 1},,status_detail_2022-03-08.csv,invalid_error_group
3,08/03: gửi lại status_detail đã sửa nhóm lỗi =...,1,{'status_detail': 1},,,
4,09-10/03: nhận 1 file xuất 2 ngày,2,,2,,


In [12]:
from src.data_prep import load_tickets

fresh = load_tickets(source="warehouse", warehouse_dir=demo.warehouse)
print("Dữ liệu tới:", fresh["time"].max().date(), "| tuổi tính tại ngày này")
display(fresh[fresh["customer_id"].isin(new_ids)][["customer_id", "usergender", "dob", "age", "age_generation"]].drop_duplicates())
fresh.loc[fresh["status_id"] == -8, ["status_id", "description", "error_group"]].drop_duplicates()

Dữ liệu tới: 2022-03-10 | tuổi tính tại ngày này


,customer_id,usergender,dob,age,age_generation
65747,900001,Female,2001-04-12,20,gen z
65885,900003,Female,2003-01-05,19,gen z
65899,900002,Male,1998-11-30,23,gen z


,status_id,description,error_group
65947,-8,3-D Secure timeout,external


- Thiếu file khách hàng → **chặn** (không đoán thông tin khách); có file → qua, và khách mới **có đủ giới tính/tuổi** trong báo cáo.
- Mã lỗi mới phải khai **nhóm lỗi hợp lệ** (customer / external / internal), nếu không mọi báo cáo theo nhóm lỗi sẽ sai → chặn tới khi sửa. Bản sửa nạp xong thì lỗi cũ tự được đánh dấu đã xử lý.
- File xuất nhiều ngày được **tự tách** theo ngày rồi kiểm tra từng ngày như bình thường.
- Nguồn gửi lại 1 file đã nạp nhưng **khác nội dung** → chặn (`file_resent_with_different_content`), không tự ghi đè dữ liệu đã báo cáo.

## 2. Báo cáo Excel tự động
Pipeline đọc **kho dữ liệu đã nạp** và sinh file Excel cho tuần 28/02/2022:

In [13]:
out_dir, summary = pipeline.run("2022-02-28", source="warehouse", paths=demo)
excel_path = out_dir / summary["excel_report"]
print("File:", excel_path.relative_to(DATA_DIR.parent))
print("Cảnh báo tuần:", summary["alerts"])

File: data\_demo\reports\weekly\2022-02-28\weekly_report_2022-02-28.xlsx
Cảnh báo tuần: [{'error_group': 'external', 'rate': 0.1776, 'baseline': 0.0867, 'z': 5.0248}]


In [14]:
from openpyxl import load_workbook
wb = load_workbook(excel_path)
pd.DataFrame([(ws.title, ws.max_row, len(ws._charts), len(ws.conditional_formatting),
               sum(1 for row in ws.iter_rows() for c in row if c.hyperlink)) for ws in wb.worksheets],
             columns=["sheet", "số dòng", "biểu đồ Excel", "quy tắc tô màu", "liên kết"])

,sheet,số dòng,biểu đồ Excel,quy tắc tô màu,liên kết
0,Tóm tắt,19,0,0,5
1,Ban lãnh đạo,51,1,1,0
2,Marketing,561,1,2,0
3,CSKH,204,0,1,0
4,Product-IT,61,1,2,0
5,Tài chính,32,1,0,0
6,Chất lượng dữ liệu,34,0,1,0


In [15]:
pd.read_excel(excel_path, sheet_name="Tóm tắt", header=None).dropna(how="all").head(20)

,0,1
0,Báo cáo vận hành tuần 2022-02-28 → 2022-03-06,NaN
1,Tạo tự động bởi src/pipeline.py · số liệu chỉ ...,NaN
3,Chỉ số,Giá trị
4,Lượt thanh toán,"1,019"
5,Tỷ lệ thanh toán thành công,80.8% (8 tuần trước: 82.4%)
6,Giao dịch lỗi,196
7,Khách mới bị mất ở bước thanh toán,148
8,Cảnh báo lỗi phía ngân hàng,CÓ - 17.8% (nền 8.7%)
9,Danh sách cứu đơn (CSKH),175 khách
10,Chất lượng dữ liệu,Đạt - file cuối ticket_history_2022-03-10__tic...


Thiết kế cho người nhận không chuyên dữ liệu:
- **Tóm tắt** ở sheet đầu, mỗi phòng ban 1 dòng tiêu điểm + **link** sang sheet của họ.
- Ô cảnh báo tô đỏ; KPI tốt/xấu hơn tháng trước tô xanh/đỏ theo **chiều tốt của từng chỉ số** (tỷ lệ lỗi giảm là tốt).
- Biểu đồ là **biểu đồ Excel gốc** (không phải ảnh) → người nhận tự lọc, sửa, copy sang slide.
- Danh sách CSKH có bộ lọc, tô đỏ các lỗi phía ngân hàng (khách không tự sửa được → ưu tiên gọi).
- Sheet **Chất lượng dữ liệu** cho biết báo cáo dựa trên dữ liệu đã qua kiểm tra. Nếu file mới nhất bị chặn, sheet Tóm tắt ghi *"Chất lượng dữ liệu: LỖI"* → người nhận biết dữ liệu mới nhất **chưa** vào báo cáo.

## 3. Dataset sạch cho analyst / product

In [16]:
tables, dictionary = marts.build()
pd.DataFrame([(name, f"{len(t):,}", t.shape[1], ", ".join(marts.TABLES[name]["key"]), marts.TABLES[name]["grain"])
              for name, t in tables.items()], columns=["bảng", "số dòng", "số cột", "khóa chính", "grain"])

,bảng,số dòng,số cột,khóa chính,grain
0,fact_payment_attempts,"154,725",28,ticket_id,1 dòng = 1 lần thanh toán (thành công hoặc lỗi)
1,dim_customer,"131,400",17,customer_id,1 dòng = 1 khách hàng trong bảng customer (kể ...
2,customer_features,"105,776",17,customer_id,"1 dòng = 1 khách đã từng mua thành công, tính ..."
3,campaign_summary,211,12,campaign_id,1 dòng = 1 campaign_id (0 = không dùng khuyến ...


In [17]:
dictionary[dictionary["table"] == "dim_customer"][["column", "dtype", "description", "null_rate", "n_unique", "examples"]]

,column,dtype,description,null_rate,n_unique,examples
28,customer_id,int64,Mã khách hàng,0.0000,131400,"100032, 100046, 100050"
29,usergender,object,Giới tính (Not verify = chưa xác minh),0.0000,3,"Female, Male, Not verify"
30,dob,datetime64[ns],Ngày sinh,0.0000,11640,"1985-08-08 00:00:00, 1987-07-11 00:00:00, 1994..."
31,age,int64,Tuổi tại ngày cuối của dữ liệu,0.0000,106,"37, 35, 28"
32,age_generation,object,Thế hệ: gen z / gen y / gen x / baby boomers,0.0000,4,"gen y, gen z, gen x"
33,first_attempt_time,datetime64[ns],Lần thanh toán đầu tiên,0.0907,119477,"2019-12-06 10:16:09.481000, 2022-01-07 13:20:2..."
34,first_success_time,datetime64[ns],Lần mua thành công đầu tiên,0.1950,105776,"2020-01-10 23:02:28.205000, 2022-01-07 13:20:2..."
35,n_attempts,int64,Tổng số lần thanh toán,0.0000,44,"3, 2, 0"
36,n_success,int64,Số lần mua thành công,0.0000,47,"2, 0, 1"
37,total_spend,float64,Tổng final_price các giao dịch thành công,0.0000,5525,"14.93, 8.95, 15.58"


Ví dụ analyst dùng ngay, không phải làm sạch lại — *"khách bị mất ở lần thanh toán đầu thuộc thế hệ nào?"* và *"nhóm khách nào có điểm quay lại cao nhất?"*:

In [18]:
dim = tables["dim_customer"]
active = dim[dim["has_activity"] & dim["is_verified_profile"]]
lost_by_gen = active.groupby("age_generation")["lost_at_first_payment"].mean().sort_values(ascending=False)
print("Tỷ lệ khách bị mất ở lần thanh toán đầu, theo thế hệ (khách có hồ sơ xác minh):")
print(lost_by_gen.map("{:.1%}".format).to_string())
tables["customer_features"].groupby("segment")["repurchase_score"].describe()[["count", "mean", "50%", "max"]].round(3)

Tỷ lệ khách bị mất ở lần thanh toán đầu, theo thế hệ (khách có hồ sơ xác minh):
age_generation
gen z           12.4%
baby boomers    12.0%
gen x           11.6%
gen y           11.1%


,count,mean,50%,max
segment,,,,
Khách giá trị cao,2675.0,0.182,0.159,0.915
Khách lâu chưa quay lại,82534.0,0.043,0.041,0.452
Khách mua một lần (mới),5810.0,0.067,0.068,0.241
Khách nhạy khuyến mãi,9185.0,0.069,0.062,0.269
Khách thường xuyên,5572.0,0.071,0.062,0.329


File xuất ra `data/marts/`: mỗi bảng có `.parquet` (giữ đúng kiểu, cho Python/BI) và `.csv` (mở bằng Excel), kèm `data_dictionary.md` / `.csv`. Khóa chính được kiểm tra không trùng trước khi ghi.

**Chạy lại toàn bộ bằng lệnh:**
```bash
python -m src.ingest simulate          # (1 lần) tạo file theo ngày
python -m src.ingest run               # nạp file mới + kiểm tra chất lượng
python -m src.pipeline --ingest        # nạp rồi ra báo cáo tuần (Excel + bản tin + CSV); lỗi dữ liệu -> dừng
python -m src.marts                    # làm mới dataset cho analyst
```